In [1]:
import pandas as pd

df = pd.read_csv("../dataset/Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## **Preprocessing**

### **Removing unnecessary columns**

In [2]:


df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [3]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### **Train-Test Split**

In [4]:
X = df.drop(['Exited'], axis=1)
y = df['Exited']

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### **Encoding categorical features and Standardization**

In [6]:
X_train['Geography'].value_counts(), X_test['Geography'].value_counts()

(Geography
 France     3994
 Germany    2011
 Spain      1995
 Name: count, dtype: int64,
 Geography
 France     1020
 Germany     498
 Spain       482
 Name: count, dtype: int64)

In [7]:
X_train['Gender'].value_counts(), X_test['Gender'].value_counts()

(Gender
 Male      4362
 Female    3638
 Name: count, dtype: int64,
 Gender
 Male      1095
 Female     905
 Name: count, dtype: int64)

In [8]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

num_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first'), ['Geography']),
    ('oe', OrdinalEncoder(), ['Gender']),
    ('scaler', StandardScaler(), num_features)
], remainder='passthrough')

X_train_preprocessed = ct.fit_transform(X_train)
X_test_preprocessed = ct.transform(X_test)


In [9]:
X_train_preprocessed[0:5]

array([[ 0.        ,  0.        ,  1.        ,  0.35649971, -0.6557859 ,
         0.34567966, -1.21847056,  0.80843615,  1.36766974,  1.        ,
         1.        ],
       [ 1.        ,  0.        ,  1.        , -0.20389777,  0.29493847,
        -0.3483691 ,  0.69683765,  0.80843615,  1.6612541 ,  1.        ,
         1.        ],
       [ 0.        ,  1.        ,  1.        , -0.96147213, -1.41636539,
        -0.69539349,  0.61862909, -0.91668767, -0.25280688,  1.        ,
         0.        ],
       [ 0.        ,  0.        ,  0.        , -0.94071667, -1.13114808,
         1.38675281,  0.95321202, -0.91668767,  0.91539272,  1.        ,
         0.        ],
       [ 0.        ,  0.        ,  1.        , -1.39733684,  1.62595257,
         1.38675281,  1.05744869, -0.91668767, -1.05960019,  0.        ,
         0.        ]])

### **Saving the preprocessor object**

In [10]:
import pickle

with open('../artifacts/preprocessor.pkl', 'wb') as f:
    pickle.dump(ct, f)

## **ANN Implementation**

In [14]:
# >>> Setting Keras Backend <<<

import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
print("Keras Backend:", keras.backend.backend())

Keras Backend: torch


In [15]:
from keras.models import Sequential
from keras.layers import Dense
from keras.callbacks import EarlyStopping, TensorBoard
import datetime

### **Building ANN model**

In [16]:
model = Sequential(
    [
        Dense(64, activation='relu', input_shape=(X_train_preprocessed.shape[1],)),    # Hidden layer 1
        Dense(32, activation='relu'),       # Hidden Layer 2
        Dense(1, activation='sigmoid')      # Output layer
    ]
)

e:\Python\conda_venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,881 (11.25 KB)

 Trainable params: 2,881 (11.25 KB)

 Non-trainable params: 0 (0.00 B)

### **Compiling model**

In [18]:
opt = keras.optimizers.Adam(learning_rate=0.01)

In [19]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy', 'auc'])

In [20]:
## >>> Early stopping <<<

early_stop_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

### **Training the model**

In [22]:
history = model.fit(X_train_preprocessed, y_train, validation_data=(X_test_preprocessed, y_test), epochs=100, callbacks=[early_stop_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 24ms/step - accuracy: 0.8364 - auc: 0.8162 - loss: 0.3866 - val_accuracy: 0.8480 - val_auc: 0.8449 - val_loss: 0.3679
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.8533 - auc: 0.8489 - loss: 0.3559 - val_accuracy: 0.8495 - val_auc: 0.8483 - val_loss: 0.3576
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 8s 33ms/step - accuracy: 0.8591 - auc: 0.8518 - loss: 0.3507 - val_accuracy: 0.8595 - val_auc: 0.8499 - val_loss: 0.3489
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step - accuracy: 0.8612 - auc: 0.8567 - loss: 0.3438 - val_accuracy: 0.8620 - val_auc: 0.8563 - val_loss: 0.3419
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 28ms/step - accuracy: 0.8609 - auc: 0.8623 - loss: 0.3395 - val_accuracy: 0.8590 - val_auc: 0.8497 - val_loss: 0.3478
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 7s 27ms/step - accuracy: 0.8596 - auc: 0.8581 - loss: 0.3423 - val_accuracy: 0.8590 - val_auc: 0.8609 - val_loss: 0.3382
Epoch 7/100
250/250 ━━━━━━━━

### **Saving the model**

In [25]:
import pickle

with open('../artifacts/model.pkl', 'wb') as f:
    pickle.dump(model, f)